# FLUX.2 Image Inpainting Demo

This notebook demonstrates how to perform **image-only inpainting** on custom regions of an image defined by point coordinates and radii using the **FLUX.2** model.

**Workflow:**
1. Configure path settings, prompt, points, and radii.
2. Load, resize, and visualize the points and affected regions.
3. Load the gated `FLUX.2-dev` model using the `HF_TOKEN` from the `.env` file.
4. Apply sequential CPU offloading and `bfloat16` data type for memory efficiency.
5. Run inpainting and visualize the results.

In [ ]:
# 1. Imports and project paths
import gc
import os
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from PIL import Image
from dotenv import load_dotenv

# Setup paths relative to the project root
cwd = Path.cwd().resolve()
if (cwd / "app" / "utils.py").is_file():
    PROJECT_ROOT = cwd
elif (cwd / "utils.py").is_file() and cwd.name == "app":
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Launch this notebook from the VideoPainter project root or its app/ directory.")

APP_DIR = PROJECT_ROOT / "app"
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

# Change directory to APP_DIR for relative utility paths
os.chdir(APP_DIR)
OUTPUT_DIR = APP_DIR / "notebook_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")

## Configuration

Set the image path, the prompt to describe the target inpainted content, coordinates for the region center(s), and the radius for each circle. Coordinates are relative to the resized resolution (default: 480x720).

In [ ]:
# 2. User configuration

# Input Image (relative to app/ or absolute path)
IMAGE_PATH = PROJECT_ROOT / "app/photo.jpg"

# Prompt describing the new object to paint into the masked area
PROMPT = "a highly detailed, clean white modern coffee ceramic mug sitting on the table"

# Manual points (x, y) coordinates on the resized image (480x720)
# Use the visualization cell below to verify and adjust these coordinates.
POINTS = [
    (240, 300),  # Center point
]

# Radius around each point to repaint (in pixels)
# Can be a single integer/float (applies to all points), or a list of radii matching the points list.
RADIUS = 60

# Model ID on Hugging Face (requires auth token for gated access)
MODEL_ID = "black-forest-labs/FLUX.2-dev"

# Output resolution (resize dimensions)
FRAME_WIDTH = 480
FRAME_HEIGHT = 720

In [ ]:
# 3. Validation and helper functions

def validate_configuration():
    if not Path(IMAGE_PATH).exists():
        raise FileNotFoundError(f"Input image not found: {IMAGE_PATH}")
    if not POINTS:
        raise ValueError("Please provide at least one coordinate in POINTS.")
    
    # Load .env file
    load_dotenv(PROJECT_ROOT / ".env")
    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        print("Warning: HF_TOKEN not found in .env file. Downloads for gated model black-forest-labs/FLUX.2-dev may fail.")
    else:
        print("HF_TOKEN loaded successfully.")
    print("Configuration validation passed.")

def create_circle_mask(width, height, points, radii):
    """Draw white circles on a black background to define inpainting regions."""
    mask_np = np.zeros((height, width), dtype=np.uint8)
    
    # Standardize radii to a list
    if not isinstance(radii, (list, tuple)):
        radii = [radii] * len(points)
        
    if len(radii) != len(points):
        raise ValueError("If RADIUS is a list, its length must match the number of POINTS.")
        
    for idx, (x, y) in enumerate(points):
        r = radii[idx]
        # Validate point coordinates are within bounds
        if not (0 <= x < width and 0 <= y < height):
            print(f"Warning: Point {idx} ({x}, {y}) is outside image boundaries ({width}x{height})")
        # Draw filled circle (thickness = -1)
        cv2.circle(mask_np, (int(x), int(y)), int(r), 255, -1)
        
    # Return as RGB PIL Image matching diffusers expected format
    return Image.fromarray(mask_np).convert("RGB")

validate_configuration()

## Visual Verification

Run the cell below to load the image and display it with red circle overlays showing the target regions. You can refine coordinates in the configuration cell above and rerun this cell until the region aligns properly.

In [ ]:
# 4. Visual verification overlay

# Load image and resize to working resolution
img_pil = Image.open(IMAGE_PATH).convert("RGB").resize((FRAME_WIDTH, FRAME_HEIGHT))
img_np = np.array(img_pil)

# Generate dynamic circle mask
mask_pil = create_circle_mask(FRAME_WIDTH, FRAME_HEIGHT, POINTS, RADIUS)

# Setup plot
fig, ax = plt.subplots(1, 2, figsize=(12, 8))

# Original image with points overlaid
ax[0].imshow(img_np)
ax[0].set_title("Original Image (Target Region Overlay)")
ax[0].axis("off")

radii_list = [RADIUS] * len(POINTS) if not isinstance(RADIUS, (list, tuple)) else RADIUS
for idx, (x, y) in enumerate(POINTS):
    r = radii_list[idx]
    # Add red bounding circle
    circle = patches.Circle((x, y), r, linewidth=2, edgecolor="r", facecolor="none", alpha=0.8)
    ax[0].add_patch(circle)
    # Add red center dot
    ax[0].plot(x, y, "ro", markersize=5)
    ax[0].text(x + 5, y - 5, f"P{idx}", color="red", fontsize=10, weight="bold")

# Generated mask
ax[1].imshow(mask_pil)
ax[1].set_title("Generated Inpainting Mask")
ax[1].axis("off")

plt.tight_layout()
plt.show()

## Load FLUX.2 and Run Inpainting

This cell loads the `Flux2Pipeline` pipeline with sequential CPU offloading and `bfloat16` data type, runs the image inpainting model on the base image and generated mask, and saves the final result.

In [ ]:
# 5. Load FLUX.2 Pipeline
from diffusers import Flux2Pipeline
from diffusers.utils import load_image

# Cleanup VRAM before loading model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Get HF token from .env
load_dotenv(PROJECT_ROOT / ".env")
hf_token = os.getenv("HF_TOKEN")

print(f"Initializing pipeline for {MODEL_ID}...")
# bfloat16 is highly recommended to maintain quality and avoid NaNs
pipe = Flux2Pipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
    token=hf_token
)

print("Applying sequential CPU offloading (saves VRAM layer-by-layer)...")
# Note: DO NOT call pipe.to("cuda") before or after this!
pipe.enable_sequential_cpu_offload()
print("FLUX.2 Pipeline loaded successfully.")

In [ ]:
# 6. Run Inpainting Inference
print("Starting FLUX.2 generation loop...")
print(f"Prompt: {PROMPT}")

with torch.inference_mode():
    output_image = pipe(
        prompt=PROMPT,
        image=img_pil,
        mask_image=mask_pil,
        num_inference_steps=30,
        guidance_scale=3.5,
        max_sequence_length=512  # Default for FLUX.2 text encoding
    ).images[0]

output_path = "flux2_inpainted_output.png"
output_image.save(output_path)
print(f"Success! Image generated and saved to {output_path}")

In [ ]:
# 7. Visualize Results
fig, ax = plt.subplots(1, 3, figsize=(18, 8))

ax[0].imshow(img_np)
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(mask_pil)
ax[1].set_title("Inpainting Mask")
ax[1].axis("off")

ax[2].imshow(output_image)
ax[2].set_title("Inpainted Output")
ax[2].axis("off")

plt.tight_layout()
plt.show()